In [3]:
import pandas as pd
from pathlib import Path

years = ['2021', '2022', '2023', '2024']

dfs = [pd.read_csv(f"atp_matches_{year}.csv") for year in years]
df = pd.concat(dfs, ignore_index=True)

print(df.shape)
print(df.info())

(11712, 49)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11712 entries, 0 to 11711
Data columns (total 49 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   tourney_id          11712 non-null  object 
 1   tourney_name        11712 non-null  object 
 2   surface             11659 non-null  object 
 3   draw_size           11712 non-null  int64  
 4   tourney_level       11712 non-null  object 
 5   tourney_date        11712 non-null  int64  
 6   match_num           11712 non-null  int64  
 7   winner_id           11712 non-null  int64  
 8   winner_seed         4929 non-null   float64
 9   winner_entry        1795 non-null   object 
 10  winner_name         11712 non-null  object 
 11  winner_hand         11712 non-null  object 
 12  winner_ht           11665 non-null  float64
 13  winner_ioc          11712 non-null  object 
 14  winner_age          11710 non-null  float64
 15  loser_id            11712 non-null  int64

In [5]:
import pandas as pd
from sklearn.utils import shuffle

# Step 1: Copy data
df_model = df.copy()

# Step 2: Drop post-match info
post_match_cols = [col for col in df_model.columns if col.startswith(('w_', 'l_'))] + ['score', 'minutes', 'winner_rank_points', 'loser_rank_points']
df_model.drop(columns=post_match_cols, inplace=True)

# Step 3: Fill and clean pre-match features
df_model['winner_seed'].fillna(0, inplace=True)
df_model['loser_seed'].fillna(0, inplace=True)
df_model['winner_entry'].fillna('N', inplace=True)
df_model['loser_entry'].fillna('N', inplace=True)

for col in ['winner_age', 'loser_age', 'winner_ht', 'loser_ht']:
    df_model[col].fillna(df_model[col].median(), inplace=True)

df_model.dropna(subset=['winner_rank', 'loser_rank'], inplace=True)

df_model['tourney_date'] = pd.to_datetime(df_model['tourney_date'], format='%Y%m%d')
df_model['year'] = df_model['tourney_date'].dt.year

round_order = {'R128':1, 'R64':2, 'R32':3, 'R16':4, 'QF':5, 'SF':6, 'F':7}
df_model['round_encoded'] = df_model['round'].map(round_order)

# Step 4: Rename columns and create symmetrical dataset
def create_symmetric_rows(row):
    p1 = {
        'player_1': row['winner_name'],
        'player_2': row['loser_name'],
        'p1_rank': row['winner_rank'],
        'p2_rank': row['loser_rank'],
        'p1_age': row['winner_age'],
        'p2_age': row['loser_age'],
        'p1_ht': row['winner_ht'],
        'p2_ht': row['loser_ht'],
        'p1_seed': row['winner_seed'],
        'p2_seed': row['loser_seed'],
        'surface': row['surface'],
        'round': row['round_encoded'],
        'year': row['year'],
        'target': 1
    }
    p2 = p1.copy()
    p2['player_1'], p2['player_2'] = p1['player_2'], p1['player_1']
    p2['p1_rank'], p2['p2_rank'] = p1['p2_rank'], p1['p1_rank']
    p2['p1_age'], p2['p2_age'] = p1['p2_age'], p1['p1_age']
    p2['p1_ht'], p2['p2_ht'] = p1['p2_ht'], p1['p1_ht']
    p2['p1_seed'], p2['p2_seed'] = p1['p2_seed'], p1['p1_seed']
    p2['target'] = 0
    return pd.DataFrame([p1, p2])

symmetric_df = pd.concat([create_symmetric_rows(row) for _, row in df_model.iterrows()], ignore_index=True)
symmetric_df = shuffle(symmetric_df, random_state=42).reset_index(drop=True)


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from sklearn.model_selection import train_test_split

# Only use numeric features + surface/round for now
feature_cols = [
    'p1_rank', 'p2_rank',
    'p1_age', 'p2_age',
    'p1_ht', 'p2_ht',
    'p1_seed', 'p2_seed',
    'round',  # encoded
    'surface'
]

# One-hot encode categorical column
X = pd.get_dummies(symmetric_df[feature_cols], columns=['surface'])
y = symmetric_df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
# Check how many NaNs are in your training set
print("NaNs in X_train:", X_train.isnull().sum().sum())
print("NaNs in y_train:", y_train.isnull().sum())


NaNs in X_train: 1521
NaNs in y_train: 0


In [20]:
# Drop rows with NaNs from X and align y accordingly
X_train_clean = X_train.dropna()
y_train_clean = y_train.loc[X_train_clean.index]

X_test_clean = X_test.dropna()
y_test_clean = y_test.loc[X_test_clean.index]

# Fit the model on cleaned data
model = LogisticRegression(max_iter=1000)
model.fit(X_train_clean, y_train_clean)

# Predict and evaluate
y_pred = model.predict(X_test_clean)
print("Accuracy:", accuracy_score(y_test_clean, y_pred))


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

Accuracy: 0.6127739806740514


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

In [22]:
round_order = {'R128': 1, 'R64': 2, 'R32': 3, 'R16': 4, 'QF': 5, 'SF': 6, 'F': 7}
df['round_encoded'] = df['round'].map(round_order)

In [25]:
# 1. Create 'player 1' = winner, 'player 2' = loser columns
df['p1_rank'] = df['winner_rank']
df['p2_rank'] = df['loser_rank']

df['p1_age'] = df['winner_age']
df['p2_age'] = df['loser_age']

df['p1_ht'] = df['winner_ht']
df['p2_ht'] = df['loser_ht']

df['p1_seed'] = df['winner_seed'].fillna(0)
df['p2_seed'] = df['loser_seed'].fillna(0)

# 2. Encode 'round'
round_order = {'R128': 1, 'R64': 2, 'R32': 3, 'R16': 4, 'QF': 5, 'SF': 6, 'F': 7}
df['round_encoded'] = df['round'].map(round_order)

# 3. One-hot encode surface
surface_dummies = pd.get_dummies(df['surface'], prefix='surface')
df = pd.concat([df, surface_dummies], axis=1)

# 4. Define the outcome (player 1 = winner)
df['outcome'] = 1


In [27]:
def create_symmetrical_df(df_original):
    rows = []
    for _, row in df_original.iterrows():
        # Player 1 = winner
        row1 = {
            'p1_rank': row['winner_rank'],
            'p2_rank': row['loser_rank'],
            'p1_age': row['winner_age'],
            'p2_age': row['loser_age'],
            'p1_ht': row['winner_ht'],
            'p2_ht': row['loser_ht'],
            'p1_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
            'p2_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
            'round_encoded': round_order.get(row['round'], 0),
            'surface': row['surface'],
            'outcome': 1
        }

        # Player 1 = loser
        row2 = {
            'p1_rank': row['loser_rank'],
            'p2_rank': row['winner_rank'],
            'p1_age': row['loser_age'],
            'p2_age': row['winner_age'],
            'p1_ht': row['loser_ht'],
            'p2_ht': row['winner_ht'],
            'p1_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
            'p2_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
            'round_encoded': round_order.get(row['round'], 0),
            'surface': row['surface'],
            'outcome': 0
        }

        rows.append(row1)
        rows.append(row2)

    return pd.DataFrame(rows)

# Create symmetrical dataset
df_symmetric = create_symmetrical_df(df)


In [33]:
# One-hot encode surface
surface_dummies = pd.get_dummies(df_symmetric['surface'], prefix='surface')
df_symmetric = pd.concat([df_symmetric, surface_dummies], axis=1)

# Define final feature list
features = ['p1_rank', 'p2_rank', 'p1_age', 'p2_age', 'p1_ht', 'p2_ht',
            'p1_seed', 'p2_seed', 'round_encoded'] + list(surface_dummies.columns)

# Prepare data
X = df_symmetric[features].dropna()
y = df_symmetric.loc[X.index, 'outcome']

# Train/test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict & evaluate
from sklearn.metrics import accuracy_score, classification_report
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

Accuracy: 0.6305386620330148
Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.65      0.63      2264
           1       0.64      0.61      0.63      2340

    accuracy                           0.63      4604
   macro avg       0.63      0.63      0.63      4604
weighted avg       0.63      0.63      0.63      4604



c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

In [34]:
import re

def parse_score(score):
    if pd.isnull(score):
        return None, None

    sets = score.strip().split()
    winner_sets = 0
    loser_sets = 0
    games_winner = 0
    games_loser = 0

    for s in sets:
        s = re.sub(r'[^\d\-]', '', s)  # strip any tiebreak junk
        if '-' not in s:
            continue
        try:
            g1, g2 = map(int, s.split('-'))
        except ValueError:
            continue

        if g1 > g2:
            winner_sets += 1
            games_winner += g1
            games_loser += g2
        else:
            loser_sets += 1
            games_winner += g2
            games_loser += g1

    sets_lost = loser_sets
    if games_winner + games_loser == 0:
        game_win_pct = None
    else:
        game_win_pct = games_winner / (games_winner + games_loser)

    return sets_lost, game_win_pct

# Apply to your original dataset
df['sets_lost'], df['game_win_pct'] = zip(*df['score'].map(parse_score))


In [36]:
from sklearn.linear_model import LinearRegression

def create_symmetrical_df(df_original):
    rows = []
    for _, row in df_original.iterrows():
        row1 = {
            'p1_rank': row['winner_rank'],
            'p2_rank': row['loser_rank'],
            'p1_age': row['winner_age'],
            'p2_age': row['loser_age'],
            'p1_ht': row['winner_ht'],
            'p2_ht': row['loser_ht'],
            'p1_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
            'p2_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
            'round_encoded': row['round_encoded'],
            'surface': row['surface'],
            'sets_lost': row['sets_lost'],
            'game_win_pct': row['game_win_pct'],
            'outcome': 1
        }
        row2 = {
            'p1_rank': row['loser_rank'],
            'p2_rank': row['winner_rank'],
            'p1_age': row['loser_age'],
            'p2_age': row['winner_age'],
            'p1_ht': row['loser_ht'],
            'p2_ht': row['winner_ht'],
            'p1_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
            'p2_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
            'round_encoded': row['round_encoded'],
            'surface': row['surface'],
            'sets_lost': 3 - row['sets_lost'] if row['sets_lost'] is not None else None,
            'game_win_pct': 1 - row['game_win_pct'] if row['game_win_pct'] is not None else None,
            'outcome': 0
        }
        rows.extend([row1, row2])
    return pd.DataFrame(rows)


In [37]:
df_symmetric = create_symmetrical_df(df)

In [38]:
y_sets_lost = df_symmetric.loc[X.index, 'sets_lost']

In [40]:
# One-hot encode surface
surface_dummies = pd.get_dummies(df_symmetric['surface'], prefix='surface')
df_symmetric = pd.concat([df_symmetric, surface_dummies], axis=1)

# Update features list
features = ['p1_rank', 'p2_rank', 'p1_age', 'p2_age',
            'p1_ht', 'p2_ht', 'p1_seed', 'p2_seed',
            'round_encoded', 'surface_Clay', 'surface_Grass', 'surface_Hard']

In [42]:
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split

# Ensure X and targets are aligned and clean
X = df_symmetric[features].dropna()

# Align targets with cleaned X
y_sets_lost = df_symmetric.loc[X.index, 'sets_lost']
y_game_pct = df_symmetric.loc[X.index, 'game_win_pct']
y_outcome = df_symmetric.loc[X.index, 'outcome']

# Drop any rows with missing targets
mask = y_sets_lost.notnull() & y_game_pct.notnull() & y_outcome.notnull()
X = X[mask]
y_sets_lost = y_sets_lost[mask]
y_game_pct = y_game_pct[mask]
y_outcome = y_outcome[mask]

# Train/test split for classifiers and regressors
X_train, X_test, y_outcome_train, y_outcome_test = train_test_split(X, y_outcome, test_size=0.2, random_state=42)
X_train_sets, X_test_sets, y_sets_train, y_sets_test = train_test_split(X, y_sets_lost, test_size=0.2, random_state=42)
X_train_games, X_test_games, y_games_train, y_games_test = train_test_split(X, y_game_pct, test_size=0.2, random_state=42)

# Train models
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_outcome_train)

sets_model = LinearRegression()
sets_model.fit(X_train_sets, y_sets_train)

games_model = LinearRegression()
games_model.fit(X_train_games, y_games_train)

# Inference function
def predict_match(input_features):
    win_prob = model.predict_proba([input_features])[0][1]
    predicted_winner = "Player 1" if win_prob > 0.5 else "Player 2"
    expected_sets_lost = sets_model.predict([input_features])[0]
    expected_game_win_pct = games_model.predict([input_features])[0]

    return {
        "predicted_winner": predicted_winner,
        "win_probability": round(win_prob, 2),
        "expected_sets_lost": round(expected_sets_lost, 2),
        "expected_game_win_pct": round(expected_game_win_pct, 2)
    }


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:767: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if not hasattr(array, "sparse") and array.dtypes.apply(is_sparse).any():
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:614: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:605: FutureWarning: is_spar

In [ ]:
import numpy as np

# Sample input row (ensure it's float type)
sample_input = X.iloc[0].values.astype(np.float64)

# Run prediction
result = predict_match(sample_input)
print(result)


{'predicted_winner': 'Player 1', 'win_probability': 0.55, 'expected_sets_lost': 1.37, 'expected_game_win_pct': 0.52}


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [48]:
df['delta_aces'] = df['w_ace'] - df['l_ace']

df['delta_1stServePct'] = (df['w_1stIn'] / df['w_svpt']) - (df['l_1stIn'] / df['l_svpt'])

df['delta_1stServeWinPct'] = (df['w_1stWon'] / df['w_1stIn']) - (df['l_1stWon'] / df['l_1stIn'])

df['delta_2ndServeWinPct'] = (df['w_2ndWon'] / (df['w_svpt'] - df['w_1stIn'])) - (df['l_2ndWon'] / (df['l_svpt'] - df['l_1stIn']))

df['delta_bpSavedPct'] = (df['w_bpSaved'] / df['w_bpFaced'].replace(0, 1)) - (df['l_bpSaved'] / df['l_bpFaced'].replace(0, 1))

df['delta_df'] = df['w_df'] - df['l_df']

df['delta_svpt'] = df['w_svpt'] - df['l_svpt']

df['delta_rank'] = df['loser_rank'] - df['winner_rank']  # higher rank = lower number, so we reverse

features = [
    'delta_aces',
    'delta_1stServePct',
    'delta_1stServeWinPct',
    'delta_2ndServeWinPct',
    'delta_bpSavedPct',
    'delta_df',
    'delta_svpt',
    'delta_rank'
]

In [52]:
import numpy as np

custom_input = np.array([
    3,    # p1_rank
    12,   # p2_rank
    26,   # p1_age
    30,   # p2_age
    188,  # p1_ht
    185,  # p2_ht
    2,    # p1_seed
    0,    # p2_seed
    5,    # round_encoded (QF)
    0,    # surface_Clay
    1,    # surface_Grass
    0     # surface_Hard
], dtype=np.float64)

result = predict_match(custom_input)
print(result)

{'predicted_winner': 'Player 1', 'win_probability': 0.55, 'expected_sets_lost': 1.39, 'expected_game_win_pct': 0.52}


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [53]:
def make_input(p1_rank, p2_rank, p1_age, p2_age, p1_ht, p2_ht,
               p1_seed, p2_seed, round_encoded, surface):
    surface_map = {'Clay': [1, 0, 0], 'Grass': [0, 1, 0], 'Hard': [0, 0, 1]}
    surface_encoded = surface_map.get(surface, [0, 0, 1])  # Default to Hard

    return np.array([
        p1_rank, p2_rank, p1_age, p2_age, p1_ht, p2_ht,
        p1_seed, p2_seed, round_encoded
    ] + surface_encoded, dtype=np.float64)

In [54]:
custom_input = make_input(
    p1_rank=3, p2_rank=12,
    p1_age=26, p2_age=30,
    p1_ht=188, p2_ht=185,
    p1_seed=2, p2_seed=0,
    round_encoded=5,  # QF
    surface='Grass'
)

result = predict_match(custom_input)
print(result)

{'predicted_winner': 'Player 1', 'win_probability': 0.55, 'expected_sets_lost': 1.39, 'expected_game_win_pct': 0.52}


c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\SAM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [58]:
print(X.var())


delta_aces                 36.867329
delta_1stServePct           0.009820
delta_1stServeWinPct        0.012536
delta_2ndServeWinPct        0.021820
delta_bpSavedPct            0.140599
delta_df                    8.958046
delta_svpt                180.775778
delta_rank              20812.675305
dtype: float64


In [69]:
def create_symmetrical_df_with_names(df_original):
    rows = []
    for _, row in df_original.iterrows():
        row1 = {
            'player_1': row['winner_name'],
            'player_2': row['loser_name'],
            'p1_rank': row['winner_rank'],
            'p2_rank': row['loser_rank'],
            'p1_age': row['winner_age'],
            'p2_age': row['loser_age'],
            'p1_ht': row['winner_ht'],
            'p2_ht': row['loser_ht'],
            'p1_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
            'p2_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
            'round_encoded': row['round_encoded'],
            'surface': row['surface'],
            'sets_lost': row['sets_lost'],
            'game_win_pct': row['game_win_pct'],
            'outcome': 1
        }
        row2 = {
            'player_1': row['loser_name'],
            'player_2': row['winner_name'],
            'p1_rank': row['loser_rank'],
            'p2_rank': row['winner_rank'],
            'p1_age': row['loser_age'],
            'p2_age': row['winner_age'],
            'p1_ht': row['loser_ht'],
            'p2_ht': row['winner_ht'],
            'p1_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
            'p2_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
            'round_encoded': row['round_encoded'],
            'surface': row['surface'],
            'sets_lost': 3 - row['sets_lost'] if row['sets_lost'] is not None else None,
            'game_win_pct': 1 - row['game_win_pct'] if row['game_win_pct'] is not None else None,
            'outcome': 0
        }
        rows.extend([row1, row2])
    return pd.DataFrame(rows)



In [70]:
df_symmetric = create_symmetrical_df_with_names(df)


In [71]:
df_symmetric['p1_id'] = df_symmetric['player_1'].astype('category').cat.codes
df_symmetric['p2_id'] = df_symmetric['player_2'].astype('category').cat.codes


In [73]:
class EloCalculator:
    def __init__(self, k=32, base_elo=1500):
        self.k = k
        self.base_elo = base_elo
        self.ratings = {}

    def get_rating(self, player):
        return self.ratings.get(player, self.base_elo)

    def expected_score(self, ra, rb):
        return 1 / (1 + 10 ** ((rb - ra) / 400))

    def update(self, winner, loser):
        ra = self.get_rating(winner)
        rb = self.get_rating(loser)

        ea = self.expected_score(ra, rb)
        eb = 1 - ea

        ra_new = ra + self.k * (1 - ea)
        rb_new = rb + self.k * (0 - eb)

        self.ratings[winner] = ra_new
        self.ratings[loser] = rb_new


In [74]:
elo = EloCalculator()

# Sort matches by date
df_sorted = df.sort_values('tourney_date')

# Store pre-match ratings
df_sorted['winner_elo'] = 0.0
df_sorted['loser_elo'] = 0.0

for i, row in df_sorted.iterrows():
    w = row['winner_name']
    l = row['loser_name']

    w_elo = elo.get_rating(w)
    l_elo = elo.get_rating(l)

    df_sorted.at[i, 'winner_elo'] = w_elo
    df_sorted.at[i, 'loser_elo'] = l_elo

    # Update Elo based on match outcome
    elo.update(winner=w, loser=l)


In [76]:
def create_symmetrical_df(df_original):
    rows = []
    for _, row in df_original.iterrows():
        row1 = {
            'player_1': row['winner_name'],
            'player_2': row['loser_name'],
            'p1_rank': row['winner_rank'],
            'p2_rank': row['loser_rank'],
            'p1_age': row['winner_age'],
            'p2_age': row['loser_age'],
            'p1_ht': row['winner_ht'],
            'p2_ht': row['loser_ht'],
            'p1_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
            'p2_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
            'round_encoded': row['round_encoded'],
            'surface': row['surface'],
            'sets_lost': row['sets_lost'],
            'game_win_pct': row['game_win_pct'],
            'outcome': 1,
            'tourney_date': row['tourney_date']
        }
        row2 = {
            'player_1': row['loser_name'],
            'player_2': row['winner_name'],
            'p1_rank': row['loser_rank'],
            'p2_rank': row['winner_rank'],
            'p1_age': row['loser_age'],
            'p2_age': row['winner_age'],
            'p1_ht': row['loser_ht'],
            'p2_ht': row['winner_ht'],
            'p1_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
            'p2_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
            'round_encoded': row['round_encoded'],
            'surface': row['surface'],
            'sets_lost': 3 - row['sets_lost'] if row['sets_lost'] is not None else None,
            'game_win_pct': 1 - row['game_win_pct'] if row['game_win_pct'] is not None else None,
            'outcome': 0,
            'tourney_date': row['tourney_date']
        }
        rows.extend([row1, row2])
    return pd.DataFrame(rows)


In [77]:
df_symmetric = df_symmetric.merge(df_sorted[['winner_name', 'tourney_date', 'winner_elo']],
                                  left_on=['player_1', 'tourney_date'],
                                  right_on=['winner_name', 'tourney_date'],
                                  how='left').rename(columns={'winner_elo': 'p1_elo'}).drop(columns='winner_name')

df_symmetric = df_symmetric.merge(df_sorted[['loser_name', 'tourney_date', 'loser_elo']],
                                  left_on=['player_2', 'tourney_date'],
                                  right_on=['loser_name', 'tourney_date'],
                                  how='left').rename(columns={'loser_elo': 'p2_elo'}).drop(columns='loser_name')

df_symmetric['delta_elo'] = df_symmetric['p1_elo'] - df_symmetric['p2_elo']


KeyError: 'tourney_date'

In [79]:
# 1. Add 'best_of' column based on tournament level
df['best_of'] = df['tourney_level'].apply(lambda x: 5 if x == 'G' else 3)

# 2. (Optional) Check distribution for sanity
print(df['best_of'].value_counts())

# 3. Include 'best_of' and 'tourney_level' in your symmetrical dataset
def create_symmetrical_df_with_format(row):
    row1 = {
        'player_1': row['winner_name'],
        'player_2': row['loser_name'],
        'p1_rank': row['winner_rank'],
        'p2_rank': row['loser_rank'],
        'p1_age': row['winner_age'],
        'p2_age': row['loser_age'],
        'p1_ht': row['winner_ht'],
        'p2_ht': row['loser_ht'],
        'p1_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
        'p2_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
        'round_encoded': round_order.get(row['round'], 0),
        'surface': row['surface'],
        'sets_lost': row['sets_lost'],
        'game_win_pct': row['game_win_pct'],
        'outcome': 1,
        'tourney_level': row['tourney_level'],
        'best_of': row['best_of'],
        'tourney_date': row['tourney_date']
    }

    row2 = {
        'player_1': row['loser_name'],
        'player_2': row['winner_name'],
        'p1_rank': row['loser_rank'],
        'p2_rank': row['winner_rank'],
        'p1_age': row['loser_age'],
        'p2_age': row['winner_age'],
        'p1_ht': row['loser_ht'],
        'p2_ht': row['winner_ht'],
        'p1_seed': row['loser_seed'] if pd.notnull(row['loser_seed']) else 0,
        'p2_seed': row['winner_seed'] if pd.notnull(row['winner_seed']) else 0,
        'round_encoded': round_order.get(row['round'], 0),
        'surface': row['surface'],
        'sets_lost': (row['best_of'] - row['sets_lost']) if row['sets_lost'] is not None else None,
        'game_win_pct': (1 - row['game_win_pct']) if row['game_win_pct'] is not None else None,
        'outcome': 0,
        'tourney_level': row['tourney_level'],
        'best_of': row['best_of'],
        'tourney_date': row['tourney_date']
    }

    return pd.DataFrame([row1, row2])

# Apply it to your dataset
df_symmetric = pd.concat([create_symmetrical_df_with_format(row) for _, row in df.iterrows()], ignore_index=True)


best_of
3    9680
5    2032
Name: count, dtype: int64


In [80]:
# Save to CSV for Excel inspection
df_symmetric.to_csv("tennis_model_data.csv", index=False)

print("✅ Exported to 'tennis_model_data.csv'")


✅ Exported to 'tennis_model_data.csv'
